# TCGA Multi-Omics Representation Learning

The goal of this study is to learn structured and biologically meaningful representations of genes through supervised node classification, and to leverage these representations to improve downstream driver gene identification. Specifically, it implements a two-stage learning framework for cancer multi-omics analysis, combining supervised representation learning with downstream driver gene prediction. The objective is to learn biologically meaningful gene representations from heterogeneous molecular data and leverage them to improve predictive performance in identifying cancer driver genes.

## 1. Stage 1: Supervised Representation Pretraining via Node Classification

In the first stage, a graph neural network operates on a biological graph defined over genes. Let the graph be denoted as $G = (V, E)$, where $V$ represents genes and $E$ represents biological interactions. Each node is associated with multi-omics features, forming a feature matrix $X \in \mathbb{R}^{N \times d}$, where $N = |V|$ is the number of genes and $d$ is the feature dimension.

The model takes as input the pair $(X, A)$, where $A \in \mathbb{R}^{N \times N}$ is the adjacency matrix of the graph. A graph neural network encoder $f_\theta$ maps these inputs to node-level embeddings:

$$
Z = f_\theta(X, A), \quad Z \in \mathbb{R}^{N \times h}
$$

where $h$ is the embedding dimension and $Z_i$ denotes the embedding of node $i$.

A prediction layer then maps embeddings to logits:

$$
z_i = g_\theta(Z_i)
$$

and corresponding probabilities are obtained via a sigmoid function:

$$
\hat{y}_i = \sigma(z_i)
$$

Each node has a binary significance label $y_i \in \{0,1\}$, and the model is trained using a node-level binary classification objective. Although the optimization target is classification, the embeddings $Z$ serve as learned representations that capture multi-omics signals, graph topology, and functional relevance.

Thus, Stage 1 performs supervised representation pretraining, where embeddings are learned implicitly through the classification task.

$$
\mathcal{L} = - \sum_i \left[
y_i \log(\hat{y}_i) + (1 - y_i)\log(1 - \hat{y}_i)
\right]
$$

## 2. Stage 2: Downstream Driver Gene Prediction

In the second stage, the embeddings learned from Stage 1 are used for driver gene prediction. This stage focuses on identifying genes that play a causal role in cancer development. The pretrained embeddings are treated as input features for a classification or ranking model.

This stage represents downstream task optimization, where the learned representations are evaluated or further utilized for a more specific biological objective. The effectiveness of the embeddings is measured by their ability to separate driver genes from non-driver genes in the learned feature space.

$$
P(\text{driver} \mid \mathbf{z}_i) = \sigma(\mathbf{W} \mathbf{z}_i + b)
$$

## 3. Evaluation Metrics

Model performance is evaluated using multiple complementary metrics, including ROC-AUC, PR-AUC, F1-score, and Precision@K. These metrics capture classification accuracy, ranking quality, and robustness under class imbalance.

$$
\text{Precision@K} = \frac{1}{K} \sum_{i=1}^{K} \mathbb{1}(y_i = 1)
$$

By integrating multi-omics data with graph-based learning, the framework aims to capture both molecular and topological signals relevant to cancer mechanisms.

$$
\theta^* = \arg\max_{\theta} \; \text{PR-AUC}_{val}
$$

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from tqdm import tqdm

import dataset, model
from dgl.dataloading import GraphDataLoader

from sklearn import metrics
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss
)
from sklearn.calibration import calibration_curve

# Jupyter display
from IPython.display import display, Image

In [ ]:
# %%
hyperparameters = {
    'num_epochs': 200,
    'out_feats': 32,
    'num_layers': 4,
    'num_heads': 1,
    'lr': 5e-4,
    'device': 'cpu'
}

omics_types = ['CNA','GE','METH','MF']
cancer_types = [
                        'BLCA',
                        'BRCA',
                        'CESC',
                        'COAD',
                        'ESCA',
                        'HNSC',
                        'KIRC',
                        'KIRP',
                        'LIHC',
                        'LUAD',
                        'LUSC',
                        'PRAD',
                        'READ',
                        'STAD',
                        'THCA',
                        'UCEC'
                    ]

# def check_raw_graphs(base=BASE_DIR):
#     for omics in ['CNA','METH','GE','MF']:
#         for cancer in ['KIRC', 'BRCA', 'READ', 'PRAD', 'STAD', 'HNSC', 'LUAD', 'THCA', 'BLCA', 'ESCA', 'LIHC', 'UCEC', 'COAD', 'LUSC', 'CESC', 'KIRP']:

# omics_types = ["MF"]
# cancer_types = ['BLCA','BRCA']


import torch.nn as nn

class FocalLoss(nn.Module):
    def __init__(self, alpha=1, gamma=2, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        # Ensure the input and target have the same shape
        if inputs.dim() > targets.dim():
            inputs = inputs.squeeze(dim=-1)
        elif targets.dim() > inputs.dim():
            targets = targets.squeeze(dim=-1)

        # Check if the shapes match after squeezing
        if inputs.size() != targets.size():
            raise ValueError(f"Target size ({targets.size()}) must be the same as input size ({inputs.size()})")

        BCE_loss = nn.functional.binary_cross_entropy_with_logits(inputs, targets, reduction='none')
        pt = torch.exp(-BCE_loss)
        F_loss = self.alpha * (1 - pt) ** self.gamma * BCE_loss

        if self.reduction == 'mean':
            return F_loss.mean()
        elif self.reduction == 'sum':
            return F_loss.sum()
        else:
            return F_loss


In [ ]:
from tqdm.auto import tqdm
import itertools

from utils import create_embedding_with_genes 
BASE_DIR = "data/graphs_multiomics"


tasks = list(itertools.product(omics_types, cancer_types))

for omics, cancer in tqdm(tasks, desc="Building graphs", ncols=100):
    create_embedding_with_genes(
        p_value=0.05,
        save=True,
        data_dir=BASE_DIR,
        omics_types=[omics],
        cancer_types=[cancer]
    )
    
print("Graph creation done")

In [ ]:
print("data_dir =", BASE_DIR)
print("type(data_dir) =", type(BASE_DIR))

In [ ]:
print("base_dir =", BASE_DIR, type(BASE_DIR))
print("omics    =", omics, type(omics))
print("cancer   =", cancer, type(cancer))

In [ ]:
# %%
import os

def check_raw_graphs(base=BASE_DIR):
    for omics in ['CNA','GE','METH','MF']:
        for cancer in [
                        'BLCA',
                        'BRCA',
                        'CESC',
                        'COAD',
                        'ESCA',
                        'HNSC',
                        'KIRC',
                        'KIRP',
                        'LIHC',
                        'LUAD',
                        'LUSC',
                        'PRAD',
                        'READ',
                        'STAD',
                        'THCA',
                        'UCEC'
                    ]:

            path = os.path.join(base, omics, cancer, "emb/raw")

            if not os.path.exists(path):
                print(f" Missing folder: {path}")
                continue

            files = os.listdir(path)

            if len(files) == 0:
                print(f"EMPTY: {omics}-{cancer}")
            else:
                print(f"{omics}-{cancer}: {files}")

check_raw_graphs()

In [ ]:
# %%
from dataset import Dataset
import os
import shutil

for omics in omics_types:
    for cancer in cancer_types:

        processed_dir = os.path.join(
            BASE_DIR,
            omics,
            cancer,
            "emb",
            "processed"
        )

        if os.path.exists(processed_dir):
            shutil.rmtree(processed_dir)
            print(f"Deleted stale processed graphs: {processed_dir}")
            
from dataset import Dataset


print("🔄 Step 3: Converting to DGL...")

for omics in ['CNA','GE','METH','MF']:
    for cancer in [
                        'BLCA',
                        'BRCA',
                        'CESC',
                        'COAD',
                        'ESCA',
                        'HNSC',
                        'KIRC',
                        'KIRP',
                        'LIHC',
                        'LUAD',
                        'LUSC',
                        'PRAD',
                        'READ',
                        'STAD',
                        'THCA',
                        'UCEC'
                    ]:

        path = os.path.join(BASE_DIR, omics, cancer)

        try:
            ds = Dataset(path)
            ds.process()  

            processed_path = os.path.join(path, "emb/processed")
            print(f"Processed: {omics}-{cancer} → {os.listdir(processed_path)}")

        except Exception as e:
            print(f"Failed processing {omics}-{cancer}: {e}")

In [ ]:


def plot_precision_at_k_curve(y_true, y_scores, max_k=200, save_path=None):

    import numpy as np
    import matplotlib.pyplot as plt

    def precision_at_k(y_true, y_scores, k):
        idx = np.argsort(y_scores)[::-1][:k]
        return np.sum(y_true[idx]) / k

    ks = list(range(1, max_k + 1))
    precisions = [precision_at_k(y_true, y_scores, k) for k in ks]

    plt.figure(figsize=(6, 4))
    plt.plot(ks, precisions, linewidth=2)
    plt.xlabel("K")
    plt.ylabel("Precision@K")
    plt.title("Precision@K Curve")
    plt.grid(False)

    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')

    plt.close()

def compare_embeddings(initial_csv, final_csv, save_dir, tag=""):

    import os
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt

    from sklearn.decomposition import PCA
    from sklearn.manifold import TSNE

    os.makedirs(save_dir, exist_ok=True)

    # =============================
    # LOAD
    # =============================
    df_init = pd.read_csv(initial_csv, index_col=0)
    df_final = pd.read_csv(final_csv, index_col=0)

    # =============================
    # ALIGN COMMON GENES
    # =============================
    common = df_init.index.intersection(df_final.index)

    df_init = df_init.loc[common]
    df_final = df_final.loc[common]

    X_init = df_init.values
    X_final = df_final.values

    print(f"🔗 Common genes: {len(common)}")

    # =============================
    # PCA COMPARISON
    # =============================
    pca = PCA(n_components=2)
    X_all = np.vstack([X_init, X_final])
    X_pca = pca.fit_transform(X_all)

    X_init_pca = X_pca[:len(X_init)]
    X_final_pca = X_pca[len(X_init):]

    plt.figure(figsize=(6,4))
    plt.scatter(X_init_pca[:,0], X_init_pca[:,1], alpha=0.4, label="Initial")
    plt.scatter(X_final_pca[:,0], X_final_pca[:,1], alpha=0.4, label="Final")

    plt.title("PCA: Initial vs Final Embeddings")
    plt.legend()
    plt.grid(False)

    plt.savefig(os.path.join(save_dir, f"pca_compare_{tag}.png"), dpi=300)
    plt.close()

    # =============================
    # t-SNE COMPARISON
    # =============================
    tsne = TSNE(n_components=2, perplexity=30, random_state=42)
    X_tsne = tsne.fit_transform(X_all)

    X_init_tsne = X_tsne[:len(X_init)]
    X_final_tsne = X_tsne[len(X_init):]

    plt.figure(figsize=(6,4))
    plt.scatter(X_init_tsne[:,0], X_init_tsne[:,1], alpha=0.4, label="Initial")
    plt.scatter(X_final_tsne[:,0], X_final_tsne[:,1], alpha=0.4, label="Final")

    plt.title("t-SNE: Initial vs Final Embeddings")
    plt.legend()
    plt.grid(False)

    plt.savefig(os.path.join(save_dir, f"tsne_compare_{tag}.png"), dpi=300)
    plt.close()

    # =============================
    # SHIFT DISTRIBUTION
    # =============================
    shifts = np.linalg.norm(X_final - X_init, axis=1)

    plt.figure(figsize=(6,4))
    plt.hist(shifts, bins=50)
    plt.title("Embedding Shift (L2 Distance)")
    plt.xlabel("Distance")
    plt.ylabel("Frequency")
    plt.grid(False)

    plt.savefig(os.path.join(save_dir, f"embedding_shift_{tag}.png"), dpi=300)
    plt.close()

    # =============================
    # SAVE NUMERICAL SUMMARY
    # =============================
    summary = {
        "mean_shift": float(np.mean(shifts)),
        "median_shift": float(np.median(shifts)),
        "max_shift": float(np.max(shifts)),
        "min_shift": float(np.min(shifts)),
    }

    pd.DataFrame([summary]).to_csv(
        os.path.join(save_dir, f"embedding_shift_stats_{tag}.csv"),
        index=False
    )

    print("✅ Embedding comparison saved.")

# =============================
# PLOTTING FUNCTIONS
# =============================

def plot_calibration(labs, probs, save_path):
    from sklearn.calibration import calibration_curve
    import matplotlib.pyplot as plt

    pt, pp = calibration_curve(labs, probs, n_bins=10)

    plt.figure(figsize=(6, 4))
    plt.plot(pp, pt, marker='o')
    plt.plot([0, 1], [0, 1], '--')
    plt.title("Calibration Curve")
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close()


def load_embeddings(initial_csv, final_csv):
    import pandas as pd
    import numpy as np

    df_init = pd.read_csv(initial_csv, index_col=0)
    df_final = pd.read_csv(final_csv, index_col=0)

    common = df_init.index.intersection(df_final.index)

    X_init = df_init.loc[common].values
    X_final = df_final.loc[common].values

    return X_init, X_final


def plot_pca(X_init, X_final, save_path):
    import numpy as np
    import matplotlib.pyplot as plt
    from sklearn.decomposition import PCA

    X_all = np.vstack([X_init, X_final])
    X_pca = PCA(n_components=2).fit_transform(X_all)

    n = len(X_init)

    plt.figure(figsize=(6, 4))
    plt.scatter(X_pca[:n, 0], X_pca[:n, 1], alpha=0.4, label="Initial")
    plt.scatter(X_pca[n:, 0], X_pca[n:, 1], alpha=0.4, label="Final")
    plt.legend()
    plt.title("PCA")
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close()


def plot_tsne(X_init, X_final, save_path):
    import numpy as np
    import matplotlib.pyplot as plt
    from sklearn.manifold import TSNE

    X_all = np.vstack([X_init, X_final])
    X_tsne = TSNE(n_components=2, random_state=42).fit_transform(X_all)

    n = len(X_init)

    plt.figure(figsize=(6, 4))
    plt.scatter(X_tsne[:n, 0], X_tsne[:n, 1], alpha=0.4, label="Initial")
    plt.scatter(X_tsne[n:, 0], X_tsne[n:, 1], alpha=0.4, label="Final")
    plt.legend()
    plt.title("t-SNE")
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close()


def plot_embedding_shift(X_init, X_final, save_path):
    import numpy as np
    import matplotlib.pyplot as plt

    shift = np.linalg.norm(X_final - X_init, axis=1)

    plt.figure(figsize=(6, 4))
    plt.hist(shift, bins=50)
    plt.title("Embedding Shift")
    plt.xlabel("L2 Distance")
    plt.ylabel("Count")
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close()

## Training and Embedding Analysis

### 1. Input and Hyperparameters

The training pipeline receives a set of hyperparameters including the number of epochs, embedding dimensionality, number of graph convolution layers, learning rate, and computation device, represented as $ \text{num\_epochs}, \ \text{out\_feats}, \ \text{num\_layers}, \ \text{lr}, \ \text{device} $. The input data consist of a multi-omics biological graph defined as $ G = (V, E, X) $, where $ V $ denotes the set of gene nodes, $ E $ represents biological interactions among genes, and $ X \in \mathbb{R}^{N \times d} $ corresponds to the node feature matrix. Each node is assigned a binary label $ y_i \in \{0,1\} $, indicating whether the gene is considered biologically significant.

### 2. Model and Representation Learning

The graph neural network learns low-dimensional node embeddings by propagating information through the graph structure. For each node, the learned embedding is computed as
$ \mathbf{z}_i = f_\theta(G, X)_i $,
where $ f_\theta $ denotes the parameterized graph neural network. These embeddings are subsequently transformed into classification logits through a linear projection:
$ \ell_i = \mathbf{W} \mathbf{z}_i + b $.
The logits are converted into probabilities using the sigmoid activation function:
$ p_i = \sigma(\ell_i) $,
which estimates the likelihood that a gene belongs to the significant class.

### 3. Training Objective

Model optimization is performed using a weighted focal loss formulation for binary classification. The objective function is defined as
$ \mathcal{L} = - \sum_i \Big[
y_i \log(p_i) + (1 - y_i)\log(1 - p_i)
\Big] $. 
This learning objective encourages the embedding space to separate biologically significant genes from non-significant genes while addressing class imbalance in the graph dataset.

### 4. Embedding Extraction

Node embeddings are extracted at multiple stages during training. Initial embeddings are obtained prior to optimization,
$ \mathbf{Z}^{(0)} = f_{\theta_0}(G, X) $,
while final embeddings are generated after convergence of the optimized model,
$ \mathbf{Z}^{(*)} = f_{\theta^*}(G, X) $.
These embeddings are saved as `.csv` files indexed by gene identifiers and are subsequently used for downstream analyses such as clustering, visualization, and biological interpretation.

### 5. Training and Validation Metrics

Throughout training and validation, multiple performance metrics are monitored to evaluate classification quality and calibration. The F1-score is computed to measure the balance between precision and recall. The Brier score, where the Brier score is defined as
$ \text{Brier} = \frac{1}{N} \sum_i (p_i - y_i)^2 $,
and quantifies the calibration quality of predicted probabilities. During optimization, the best-performing model is selected according to the maximum validation PR-AUC criterion,
$ \max \text{PR-AUC} $,
which is particularly appropriate for highly imbalanced biological classification tasks.
These embeddings serve as pretrained representations for downstream driver gene prediction.

In [ ]:
def train(hyperparams=None, data_path='data', plot=True, omics='mf', cancer='KIRC'):

    import os, copy, pickle
    import numpy as np
    import pandas as pd
    import torch
    import torch.optim as optim
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt

    from tqdm import tqdm
    from sklearn import metrics
    from sklearn.metrics import (
        roc_auc_score,
        average_precision_score,
        precision_score,
        recall_score,
        brier_score_loss
    )
    from sklearn.calibration import calibration_curve
    from sklearn.decomposition import PCA
    from sklearn.manifold import TSNE

    # =============================
    # HYPERPARAMS
    # =============================
    num_epochs = hyperparams['num_epochs']
    out_feats = hyperparams['out_feats']
    num_layers = hyperparams['num_layers']
    lr = hyperparams['lr']
    device = hyperparams['device']

    tag = f"lr{lr:.0e}_dim{out_feats}_lay{num_layers}_epo{num_epochs}"

    # =============================
    # PATHS
    # =============================
    model_dir = os.path.join(data_path, omics, cancer, 'emb/models')
    os.makedirs(model_dir, exist_ok=True)

    results_path = os.path.join("process/graphs_multiomics", omics, cancer)
    os.makedirs(results_path, exist_ok=True)

    model_path = os.path.join(model_dir, f"model_{tag}.pth")

    # =============================
    # DATA
    # =============================
    ds = dataset.Dataset(os.path.join(data_path, omics, cancer))
    print('os.path.join(data_path, omics, cancer)===================', os.path.join(data_path, omics, cancer))
    dl_train = GraphDataLoader([ds[0]], batch_size=1)
    dl_valid = GraphDataLoader([ds[1]], batch_size=1)

    # =============================
    # MODEL
    # =============================
    net = model.TAGCNModel(out_feats, num_layers, do_train=True).to(device)
    optimizer = optim.Adam(net.parameters(), lr=lr)
    best_model = copy.deepcopy(net)

    criterion = FocalLoss(alpha=0.25, gamma=2.0)
    weight = torch.tensor([0.00001, 0.99999]).to(device)

    # =============================
    # SAVE EMBEDDINGS FUNCTION
    # =============================
    def save_embeddings(model, dl, save_path):

        model.eval()

        for graph, _ in dl:
            graph = graph.to(device)

            emb = model.get_node_embeddings(graph).detach().cpu().numpy()

            graph_path = os.path.join(data_path, omics, cancer, 'emb/raw/emb_train.pkl')
            nx_graph = pickle.load(open(graph_path, 'rb'))

            node_to_index = {n: i for i, n in enumerate(nx_graph.nodes)}

            stid_dict = {}
            for node in nx_graph.nodes:
                if 'stId' in nx_graph.nodes[node]:
                    stid = nx_graph.nodes[node]['stId']
                    stid_dict[stid] = emb[node_to_index[node]]

            df = pd.DataFrame.from_dict(stid_dict, orient='index')
            df.to_csv(save_path, index_label='stId')

            print(f"Saved embeddings → {save_path}")
            break

    # =============================
    # SAVE INITIAL EMBEDDINGS
    # =============================
    initial_csv = os.path.join(results_path, f"embeddings_{tag}_initial.csv")
    print(initial_csv)
    print('initial_csv**************************************', initial_csv)
    save_embeddings(net, dl_train, initial_csv)

    # =============================
    # HISTORY
    # =============================
    history = {
        "train_loss": [], "val_loss": [],
        "train_f1": [], "val_f1": [],
        "val_auc": [], "val_pr_auc": [],
        "val_brier": []
    }

    best_pr_auc = 0.0

    # =============================
    # TRAIN LOOP
    # =============================
    for epoch in tqdm(range(num_epochs), desc=f"{omics}-{cancer}"):

        # -------- TRAIN --------
        net.train()
        tl, tp, tlab = [], [], []

        for g, _ in dl_train:
            g = g.to(device)

            logits = net(g)
            labels = g.ndata['significance'].unsqueeze(-1)

            weight_ = weight[labels.view(-1).long()].view_as(labels)

            loss = criterion(logits, labels)
            loss = (loss * weight_).mean()

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            tl.append(loss.item())

            probs = logits.sigmoid().detach().cpu().numpy()
            preds = (probs > 0.3).astype(int)

            tp.extend(preds.flatten())
            tlab.extend(labels.cpu().numpy().flatten())

        history["train_loss"].append(np.mean(tl))
        history["train_f1"].append(metrics.f1_score(tlab, tp))

        # -------- VALID --------
        net.eval()
        vl, probs, labs = [], [], []

        with torch.no_grad():
            for g, _ in dl_valid:
                g = g.to(device)

                logits = net(g)
                labels = g.ndata['significance'].unsqueeze(-1)

                weight_ = weight[labels.view(-1).long()].view_as(labels)

                loss = criterion(logits, labels)
                loss = (loss * weight_).mean()

                vl.append(loss.item())

                p = logits.sigmoid().cpu().numpy()
                probs.extend(p.flatten())
                labs.extend(labels.cpu().numpy().flatten())

        probs = np.array(probs)
        labs = np.array(labs)

        history["val_loss"].append(np.mean(vl))
        history["val_f1"].append(metrics.f1_score(labs, (probs > 0.3)))
        history["val_auc"].append(roc_auc_score(labs, probs))
        history["val_pr_auc"].append(average_precision_score(labs, probs))
        history["val_brier"].append(brier_score_loss(labs, probs))

        if history["val_pr_auc"][-1] > best_pr_auc:
            best_pr_auc = history["val_pr_auc"][-1]
            best_model.load_state_dict(copy.deepcopy(net.state_dict()))

    # =============================
    # SAVE MODEL
    # =============================
    torch.save(best_model.state_dict(), model_path)

    # =============================
    # SAVE FINAL EMBEDDINGS
    # =============================
    final_csv = os.path.join(results_path, f"embeddings_{tag}_final.csv")
    save_embeddings(best_model, dl_train, final_csv)

    # =============================
    # SAVE METRICS
    # =============================
    pd.DataFrame(history).to_csv(
        os.path.join(results_path, f"metrics_{tag}.csv"),
        index=False
    )

    # ====================================
    # SAVE TRAINING CURVES
    # ====================================

    metrics_to_plot = {

        "train_loss":"Loss",
        "val_loss":"Loss",

        "train_f1":"F1",
        "val_f1":"F1",

        "val_auc":"AUROC",

        "val_pr_auc":"PR-AUC",

        "val_brier":"Brier Score"
    }

    for metric, ylabel in metrics_to_plot.items():

        if metric not in history:
            continue

        plt.figure(figsize=(8,6))

        plt.plot(
            history[metric],
            linewidth=3
        )

        plt.xlabel(
            "Epoch",
            fontsize=14
        )

        plt.ylabel(
            ylabel,
            fontsize=14
        )

        plt.title(
            metric.replace(
                "_",
                " "
            ).title(),
            fontsize=16
        )

        plt.tight_layout()

        save_name = metric.replace(
            "train_",
            ""
        )

        plt.savefig(

            os.path.join(

                results_path,
                f"{save_name}_{tag}.png"

            ),

            dpi=300,
            bbox_inches='tight'
        )

        plt.show()


        # =============================
        # SAVE Precision@K CURVE
        # =============================
        if plot:
            plot_precision_at_k_curve(
                labs,
                probs,
                save_path=os.path.join(results_path, f"p_at_k_{tag}.png")
            )
            
        with open(os.path.join(results_path, f"summary_{tag}.txt"), "w") as f:
            f.write(f"Best PR-AUC: {best_pr_auc}\n")

        # # =============================
        # # PLOTS
        # # =============================
        # if plot:

        #     # Calibration
        #     plot_calibration(
        #         labs, probs,
        #         save_path=os.path.join(results_path, f"calibration_{tag}.png")
        #     )

        #     # Embedding comparison
        #     X_init, X_final = load_embeddings(initial_csv, final_csv)

        #     plot_pca(
        #         X_init, X_final,
        #         save_path=os.path.join(results_path, f"pca_{tag}.png")
        #     )

        #     plot_tsne(
        #         X_init, X_final,
        #         save_path=os.path.join(results_path, f"tsne_{tag}.png")
        #     )

        #     plot_embedding_shift(
        #         X_init, X_final,
        #         save_path=os.path.join(results_path, f"shift_{tag}.png")
        #     )



        # =============================
            # SAVE METRICS
            # =============================
            metrics_csv = os.path.join(
                results_path,
                f"metrics_{tag}.csv"
            )

            pd.DataFrame(history).to_csv(
                metrics_csv,
                index=False
            )

            print("Saved:", metrics_csv)

            # ====================================
            # SAVE TRAINING CURVES
            # ====================================

            plot_groups = [

                (
                    ["train_loss","val_loss"],
                    "Loss",
                    f"loss_{tag}.png"
                ),

                (
                    ["train_f1","val_f1"],
                    "F1",
                    f"f1_{tag}.png"
                ),

                (
                    ["val_auc"],
                    "AUROC",
                    f"roc_auc_{tag}.png"
                ),

                (
                    ["val_pr_auc"],
                    "PR-AUC",
                    f"pr_auc_{tag}.png"
                ),

                (
                    ["val_brier"],
                    "Brier Score",
                    f"brier_{tag}.png"
                )
            ]

            for metrics_list, ylabel, save_name in plot_groups:

                plt.figure(figsize=(8,6))

                for metric in metrics_list:

                    plt.plot(
                        history[metric],
                        linewidth=3,
                        label=metric
                    )

                plt.xlabel(
                    "Epoch",
                    fontsize=14
                )

                plt.ylabel(
                    ylabel,
                    fontsize=14
                )

                plt.legend()

                plt.tight_layout()

                save_path=os.path.join(
                    results_path,
                    save_name
                )

                plt.savefig(
                    save_path,
                    dpi=300,
                    bbox_inches="tight"
                )

                plt.close()

                print("Saved:", save_path)

            # =============================
            # SAVE SUMMARY
            # =============================

            summary_path=os.path.join(
                results_path,
                f"summary_{tag}.txt"
            )

            with open(summary_path,"w") as f:

                f.write(
                    f"Best PR-AUC: {best_pr_auc:.4f}\n"
                )

            print("Saved:",summary_path)

            # =============================
            # EXTRA PLOTS
            # =============================

            if plot:

                print("Generating additional plots...")

                plot_precision_at_k_curve(
                    labs,
                    probs,
                    save_path=os.path.join(
                        results_path,
                        f"p_at_k_{tag}.png"
                    )
                )

                plot_calibration(
                    labs,
                    probs,
                    save_path=os.path.join(
                        results_path,
                        f"calibration_{tag}.png"
                    )
                )

                X_init,X_final=load_embeddings(
                    initial_csv,
                    final_csv
                )

                plot_pca(
                    X_init,
                    X_final,
                    save_path=os.path.join(
                        results_path,
                        f"pca_{tag}.png"
                    )
                )

                plot_tsne(
                    X_init,
                    X_final,
                    save_path=os.path.join(
                        results_path,
                        f"tsne_{tag}.png"
                    )
                )

                plot_embedding_shift(
                    X_init,
                    X_final,
                    save_path=os.path.join(
                        results_path,
                        f"shift_{tag}.png"
                    )
                )

            print(f"Done: {omics}-{cancer}")

            return model_path





In [ ]:
import os

print(os.getcwd())

In [ ]:
all_results = {}

for omics in omics_types:
    for cancer in cancer_types:

        print(f"\n=== {omics} | {cancer} ===")

        try:
            # 🔍 check processed graphs BEFORE training
            processed_dir = os.path.join(
                BASE_DIR, omics, cancer, "emb/processed"
            )

            if not os.path.exists(processed_dir) or len(os.listdir(processed_dir)) < 2:
                print(f"⚠️ Skipping {omics}-{cancer}: no processed graphs")
                continue

            model_path = train(
                hyperparams=hyperparameters,
                data_path=BASE_DIR,
                plot=True,
                omics=omics,
                cancer=cancer
            )

            results_path = os.path.join(
                "process/graphs_multiomics",
                omics,
                cancer
            )

            all_results[(omics, cancer)] = results_path

        except Exception as e:
            print(f"❌ Failed: {omics}-{cancer}: {e}")

# all_results = {}

# for omics in omics_types:
#     for cancer in cancer_types:

#         print(f"\n=== {omics} | {cancer} ===")

#         try:
#             model_path = train(
#                 hyperparams=hyperparameters,
#                 data_path="process/graphs_multiomics",
#                 plot=True,
#                 omics=omics,
#                 cancer=cancer
#             )

#             results_path = os.path.join(
#                 "results/embeddings",
#                 omics,
#                 cancer
#             )

#             tag = f"dim{hyperparameters['out_feats']}_lay{hyperparameters['num_layers']}_epo{hyperparameters['num_epochs']}"

#             all_results[(omics, cancer)] = results_path

#         except Exception as e:
#             print(f"Failed: {omics}-{cancer}: {e}")

In [ ]:
import os

import os
import pandas as pd
import re


def extract_tag(filename):

    match = re.search(
        r"(lr.*?_dim.*?_lay.*?_epo.*?)\.csv",
        filename
    )

    return match.group(1) if match else None


def find_best_run(results_path):

    print("\nSearching in:")
    print(results_path)

    files=os.listdir(results_path)

    print("\nFound files:")
    for f in files:
        print(f)

    best_score=-1
    best_tag=None
    best_file=None

    for file in files:

        if not (
            file.startswith("metrics_")
            and file.endswith(".csv")
        ):
            continue

        full_path=os.path.join(
            results_path,
            file
        )

        print("\nReading:")
        print(file)

        try:

            df=pd.read_csv(
                full_path
            )

            print(
                "Columns:",
                df.columns.tolist()
            )

            if "val_pr_auc" not in df.columns:

                print(
                    "Missing val_pr_auc"
                )
                continue

            score=df[
                "val_pr_auc"
            ].max()

            print(
                "Best PR:",
                score
            )

            tag=extract_tag(
                file
            )

            if score>best_score:

                best_score=score
                best_tag=tag
                best_file=file

        except Exception as e:

            print(
                "ERROR:",
                e
            )

    return (
        best_tag,
        best_score,
        best_file
    )

def find_exact_embeddings(results_path, tag):

    init_file = None
    final_file = None

    for f in os.listdir(results_path):

        # flexible match
        if tag in f and "initial" in f and f.endswith(".csv"):
            init_file = os.path.join(results_path, f)

        if tag in f and "final" in f and f.endswith(".csv"):
            final_file = os.path.join(results_path, f)

    if init_file is None:
        print(f"Missing initial for tag: {tag}")

    if final_file is None:
        print(f"Missing final for tag: {tag}")

    return init_file, final_file



def plot_tsne_before_after(initial_csv, final_csv, save_path=None, perplexity=30):

    import pandas as pd
    import numpy as np
    import matplotlib.pyplot as plt
    from sklearn.manifold import TSNE

    # -----------------------------
    # Load embeddings
    # -----------------------------
    init = pd.read_csv(initial_csv, index_col=0)
    final = pd.read_csv(final_csv, index_col=0)

    # Ensure same ordering
    common_idx = init.index.intersection(final.index)
    init = init.loc[common_idx]
    final = final.loc[common_idx]

    # -----------------------------
    # t-SNE (fit separately)
    # -----------------------------
    tsne_init = TSNE(
        n_components=2,
        perplexity=perplexity,
        random_state=42,
        n_iter=1000
    ).fit_transform(init.values)

    tsne_final = TSNE(
        n_components=2,
        perplexity=perplexity,
        random_state=42,
        n_iter=1000
    ).fit_transform(final.values)



results_path = os.path.join(
    "process/graphs_multiomics",
    omics,
    cancer
)

tag = f"lr{hyperparameters['lr']:.0e}_dim{hyperparameters['out_feats']}_lay{hyperparameters['num_layers']}_epo{hyperparameters['num_epochs']}"

initial_csv, final_csv = find_exact_embeddings(results_path, tag)

if initial_csv and final_csv:
    plot_tsne_before_after(
        initial_csv,
        final_csv,
        save_path=os.path.join(results_path, f"tsne_{tag}.png")
    )
else:
    print("Exact match not found")



In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def compare_embeddings(initial_csv, final_csv, save_path=None, tag=None):

    init = pd.read_csv(initial_csv, index_col=0)
    final = pd.read_csv(final_csv, index_col=0)

    # -----------------------------
    # align (important for correctness)
    # -----------------------------
    common_idx = init.index.intersection(final.index)
    init = init.loc[common_idx]
    final = final.loc[common_idx]

    # -----------------------------
    # L2 shift per gene/node
    # -----------------------------
    shift = np.linalg.norm(final.values - init.values, axis=1)

    # -----------------------------
    # plot
    # -----------------------------
    plt.figure(figsize=(6, 4))
    plt.hist(shift, bins=50)

    plt.title("Embedding Shift Distribution")
    plt.xlabel("L2 Distance")
    plt.ylabel("Count")

    plt.grid(False)

    # -----------------------------
    # save path logic
    # -----------------------------
    if save_path is None and tag is not None:
        results_dir = os.path.dirname(initial_csv)

        save_path = os.path.join(
            results_dir,
            f"embedding_shift_{tag}.png"
        )

    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches="tight")

    plt.close()

    return shift

# initial_csv, final_csv = find_exact_embeddings(results_path, tag)
results_path = os.path.join(
    "process/graphs_multiomics",
    omics,
    cancer
)

tag = f"lr{hyperparameters['lr']:.0e}_dim{hyperparameters['out_feats']}_lay{hyperparameters['num_layers']}_epo{hyperparameters['num_epochs']}"

initial_csv, final_csv = find_exact_embeddings(results_path, tag)

compare_embeddings(
    initial_csv,
    final_csv,
    save_path=os.path.join(results_path, f"shift_{tag}.png")
)

In [ ]:
initial_csv

In [ ]:
# %%
from IPython.display import display, Image
import os
import pandas as pd
import re


# def extract_tag(filename):
#     match = re.search(r"(lr\d+_dim\d+_lay\d+_epo\d+)", filename)
#     return match.group(1) if match else None


# def find_best_run(results_path):

#     best_score = -1
#     best_tag = None
#     best_file = None

#     for file in os.listdir(results_path):
#         if file.startswith("metrics_") and file.endswith(".csv"):

#             tag = extract_tag(file)
#             full_path = os.path.join(results_path, file)

#             try:
#                 df = pd.read_csv(full_path)

#                 if "val_pr_auc" not in df.columns:
#                     continue

#                 score = df["val_pr_auc"].max()

#                 if score > best_score:
#                     best_score = score
#                     best_tag = tag
#                     best_file = file

#             except Exception as e:
#                 print(f"Skipping {file}: {e}")

#     return best_tag, best_score, best_file


def display_best_plots(results_path, tag):

    if tag is None:
        print("No valid run found")
        return

    print(f"BEST TAG: {tag}")

    plot_types=[

        "loss",
        "f1",

        "val_auc",
        "val_pr_auc",
        "val_brier",

        "calibration",
        "p_at_k",
        "shift",
        "pca",
        "tsne"
    ]

    for ptype in plot_types:
        found = False

        for file in os.listdir(results_path):
            if ptype in file and tag in file and file.endswith(".png"):
                print(f"[{ptype}] {file}")

                # OPTIONAL: control display size (NOT plot size)
                display(Image(filename=os.path.join(results_path, file), width=500))

                found = True
                break

        if not found:
            print(f"Missing plot: {ptype}")


# ================================
# 🔥 MAIN LOOP
# ================================
for (omics, cancer), path in all_results.items():

    print(f"\n==============================")
    print(f"{omics} | {cancer}")
    print(f"==============================")

    if not os.path.exists(path):
        print("❌ Path not found")
        continue

    # -----------------------------
    # STEP 1: find best run
    # -----------------------------
    best_tag, best_score, best_file = find_best_run(path)

    if best_tag is None:
        print("❌ No valid training_metrics found")
        continue

    print(f"BEST RUN: {best_tag}")
    print(f"BEST PR-AUC: {best_score:.4f}")
    print(f"Source CSV: {best_file}")

    # -----------------------------
    # STEP 2: display plots
    # -----------------------------
    display_best_plots(path, best_tag)

In [ ]:
import os

print("\n=== OUTPUT FILES IN RESULTS DIR ===\n")
for f in os.listdir(results_path):
    if any(x in f for x in ["p_at_k", "shift", "tsne"]):
        print(f)

In [ ]:
import os
import pandas as pd
from tqdm.auto import tqdm

# =========================================================
# CONFIG 
# =========================================================
omics_types = ['CNA','GE','METH','MF']
cancer_types = [
                        'BLCA',
                        'BRCA',
                        'CESC',
                        'COAD',
                        'ESCA',
                        'HNSC',
                        'KIRC',
                        'KIRP',
                        'LIHC',
                        'LUAD',
                        'LUSC',
                        'PRAD',
                        'READ',
                        'STAD',
                        'THCA',
                        'UCEC'
                    ]


OUTPUT_PATH = os.path.join("process/graphs_multiomics", "combined_omics_embeddings_32x16x4.tsv")

TAG = f"lr{hyperparameters['lr']:.0e}_dim{hyperparameters['out_feats']}_lay{hyperparameters['num_layers']}_epo{hyperparameters['num_epochs']}"

# =========================================================
# MERGE
# =========================================================
combined_df = None

for omics in omics_types:
    for cancer in tqdm(cancer_types, desc=f"{omics}"):

        file_path = os.path.join(
            "process/graphs_multiomics",
            omics,
            cancer,
            f"embeddings_{TAG}_final.csv"
        )

        if not os.path.exists(file_path):
            print(f"⚠️ Missing: {file_path}")
            continue

        df = pd.read_csv(file_path, index_col=0)

        # ----------------------------------------
        # Rename columns → avoid collisions
        # ----------------------------------------
        df.columns = [
            f"{omics}_{cancer}_dim{i}"
            for i in range(df.shape[1])
        ]

        # ----------------------------------------
        # Merge (outer join on genes)
        # ----------------------------------------
        if combined_df is None:
            combined_df = df
        else:
            combined_df = combined_df.join(df, how="outer")

# =========================================================
# CLEANUP
# =========================================================
combined_df = combined_df.sort_index()

print("Final shape:", combined_df.shape)

# Optional: fill missing embeddings
combined_df = combined_df.fillna(0)

# =========================================================
# SAVE
# =========================================================
combined_df.to_csv(OUTPUT_PATH)

print(f"Saved combined embeddings → {OUTPUT_PATH}")

In [ ]:
!pwd